In [108]:
import ast
import csv
import json
import random
from pathlib import Path
import zipfile
from collections import OrderedDict
from pathlib import Path
from typing import Any

import torch



In [109]:
PROJECT_ROOT = Path("C:/Users/ragha/OneDrive/Desktop/SDP-Sequential-Diagnosis/AFA_bench")

DDX_zip = PROJECT_ROOT.parent / "data" / "release_train_patients.zip"
evidence_json = PROJECT_ROOT.parent / "data" / "release_evidences.json"

PROJECT_ROOT / "extra" / "data" / "misc"
pain_feature = "pain"
PAIN_FEATURE = pain_feature

N_patients = 20_000
selection_seed = 42
split_seed = 42

train_frac = 0.80
val_frac = 0.10
test_frac = 0.10

E53 = "E_53"
E55 = "E_55"

EXCLUDED_FIELDS = {
    "PATHOLOGY",
    "DIFFERENTIAL_DIAGNOSIS",
    "EVIDENCES",
    "INITIAL_EVIDENCE",
}


### utilities


In [110]:
def parse_literal(value: str) -> Any:
    value = value.strip()
    if not value:
        return None
    try:
        return ast.literal_eval(value)
    except (ValueError, SyntaxError) as exc:
        raise ValueError(f"Could not parse DDXPlus value: {value[:200]}") from exc

def load_raw_patients() -> list[dict[str, str]]:
    if not DDX_zip.exists():
        raise FileNotFoundError(f"Missing DDXPlus ZIP: {DDX_ZIP}")
    if not evidence_json.exists():
        raise FileNotFoundError(f"Missing evidence definitions: {EVIDENCE_JSON}")

    with zipfile.ZipFile(DDX_zip) as zf:
        names = [name for name in zf.namelist() if not name.endswith("/")]
        if len(names) != 1:
            raise ValueError(
                f"Expected exactly one file in the ZIP, found: {names}"
            )

        with zf.open(names[0]) as raw_file:
            text = raw_file.read().decode("utf-8")

    rows = list(csv.DictReader(text.splitlines()))
    required = {
        "AGE",
        "SEX",
        "PATHOLOGY",
        "EVIDENCES",
        "INITIAL_EVIDENCE",
        "DIFFERENTIAL_DIAGNOSIS",
    }
    missing = required - set(rows[0].keys()) if rows else required
    if missing:
        raise ValueError(f"Missing raw DDXPlus columns: {sorted(missing)}")
    return rows

def load_evidence_definitions() -> dict[str, dict[str, Any]]:
    with evidence_json.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, dict):
        raise TypeError("release_evidences.json must contain an object/dict")
    return data

def select_patients(rows):
    rng = random.Random(selection_seed)

    indices = list(range(len(rows)))
    rng.shuffle(indices)

    return [rows[i] for i in indices[:N_patients]]


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)


### feature cleaning


In [111]:
from collections import OrderedDict

ORDERED_EVIDENCE_IDS = {
    "E_59": "How fast did the pain appear?",
    "E_56": "How intense is the pain?",
    "E_58": "How precisely is the pain located?",
    "E_134": "How intense is the pain caused by the rash?",
    "E_136": "How severe is the itching?",
}


def is_ordered_evidence(evidence_id, definition):
    """Return True only for the verified ordered DDXPlus variables above."""
    if evidence_id not in ORDERED_EVIDENCE_IDS:
        return False
    if definition.get("data_type") != "C":
        return False

    values = definition.get("possible-values", [])
    try:
        nums = [int(v) for v in values]
    except (TypeError, ValueError):
        return False

    return nums == list(range(0, 11))


def build_feature_vocabulary(selected_rows, evidence_defs):
    
    features = OrderedDict()
    features["age"] = None
    features["sex"] = None
    features[PAIN_FEATURE] = None

    for evidence_id, definition in evidence_defs.items():
        data_type = definition.get("data_type")

        if evidence_id in (E53, E55):
            continue

        if data_type == "B":
            features[evidence_id] = None

        elif data_type == "M":
            for value_id in definition.get("possible-values", []):
                features[f"{evidence_id}@{value_id}"] = None

        elif data_type == "C":
            if is_ordered_evidence(evidence_id, definition):
                for value in definition.get("possible-values", []):
                    features[f"{evidence_id}_level_{int(value)}"] = None
            else:
                for value_id in definition.get("possible-values", []):
                    features[f"{evidence_id}@{value_id}"] = None

    return ["age", "sex", "pain"] + sorted(
        name for name in features if name not in ("age", "sex", "pain")
    )


### feature representation

In [112]:
def extract_patient_evidence(row):
    value = parse_literal(row["EVIDENCES"])

    if not isinstance(value, list):
        raise TypeError("EVIDENCES must parse to a list.")

    return [x for x in value if isinstance(x, str)]


def pain_is_present(row):
    evidence = extract_patient_evidence(row)

    if any(item == E53 or item.startswith(E53 + "_@_") for item in evidence):
        return True

    for item in evidence:
        if item.startswith(E55 + "_@_"):
            _, value_id = item.split("_@_", 1)
            if value_id != "V_123":
                return True
        elif item == E55:
            return True

    return False


def encode_sex(sex):
    mapping = {"F": 0.0, "M": 1.0}
    sex = sex.strip().upper()
    if sex not in mapping:
        raise ValueError(f"Unexpected SEX value: {sex!r}")
    return mapping[sex]


def encode_row(row, feature_names, evidence_defs):
    values = {name: 0.0 for name in feature_names}

    # Real numerical quantity.
    values["age"] = float(row["AGE"])

    # Binary demographic feature.
    values["sex"] = encode_sex(row["SEX"])

    # Canonical binary pain feature.
    values[PAIN_FEATURE] = 1.0 if pain_is_present(row) else 0.0

    for item in extract_patient_evidence(row):
        if "_@_" in item:
            evidence_id, value_id = item.split("_@_", 1)
        else:
            evidence_id, value_id = item, None

        if evidence_id in (E53, E55) or evidence_id not in evidence_defs:
            continue

        definition = evidence_defs[evidence_id]
        data_type = definition.get("data_type")

        if data_type == "C":
            if is_ordered_evidence(evidence_id, definition):
                # Preserve the actual ordered level as a one-hot binary feature.
                try:
                    level = int(value_id)
                except (TypeError, ValueError) as exc:
                    raise ValueError(
                        f"Expected numeric severity value for {evidence_id}, got {value_id!r}"
                    ) from exc
                name = f"{evidence_id}_level_{level}"
            else:
                name = f"{evidence_id}@{value_id}"

        elif data_type == "M":
            name = f"{evidence_id}@{value_id}"

        else:
            # Binary evidence: the raw evidence ID is the feature name.
            name = evidence_id

        if name in values:
            values[name] = 1.0

    return [values[name] for name in feature_names]


### train normalization

In [113]:
def normalize_age(train_x, other_x, feature_names):
    age_idx = feature_names.index("age")

    mean = train_x[:, age_idx].mean().item()
    std = train_x[:, age_idx].std(unbiased=False).item()
    if std == 0:
        std = 1.0

    train_x = train_x.clone()
    train_x[:, age_idx] = (train_x[:, age_idx] - mean) / std

    normalized_other = []
    for x in other_x:
        x = x.clone()
        x[:, age_idx] = (x[:, age_idx] - mean) / std
        normalized_other.append(x)

    return train_x, normalized_other, mean, std


### save final dataset

In [114]:
import pandas as pd


def build_encoding_metadata(evidence_defs):
    metadata = {
        "ordered_evidence": {},
        "categorical_evidence": {},
        "binary_evidence": [],
        "multivalue_evidence": {},
    }

    for evidence_id, definition in evidence_defs.items():
        if evidence_id in (E53, E55):
            continue

        data_type = definition.get("data_type")
        if data_type == "B":
            metadata["binary_evidence"].append(evidence_id)
        elif data_type == "M":
            metadata["multivalue_evidence"][evidence_id] = definition.get("possible-values", [])
        elif data_type == "C":
            if is_ordered_evidence(evidence_id, definition):
                metadata["ordered_evidence"][evidence_id] = {
                    "question_en": definition.get("question_en", ""),
                    "levels": [int(v) for v in definition.get("possible-values", [])],
                    "encoding": "one-hot binary level features",
                }
            else:
                metadata["categorical_evidence"][evidence_id] = {
                    "question_en": definition.get("question_en", ""),
                    "values": definition.get("possible-values", []),
                }

    return metadata


def make_clean_dataframe(rows, feature_names, label_mapping, evidence_defs):
    data = []

    for row in rows:
        features = encode_row(row, feature_names, evidence_defs)
        label = label_mapping[row["PATHOLOGY"].strip()]
        data.append(features + [label])

    return pd.DataFrame(
        data,
        columns=feature_names + ["pathology"],
    )


def main():
    rows = load_raw_patients()
    evidence_defs = load_evidence_definitions()

    selected_rows = select_patients(rows)

    feature_names = build_feature_vocabulary(
        selected_rows,
        evidence_defs,
    )

    label_mapping = build_label_mapping(selected_rows)
    encoding_metadata = build_encoding_metadata(evidence_defs)

    df = make_clean_dataframe(
        selected_rows,
        feature_names,
        label_mapping,
        evidence_defs,
    )

    output_root.mkdir(parents=True, exist_ok=True)

    df.to_csv(
        output_root / "ddxplus.csv",
        index=False,
    )

    save_json(
        label_mapping,
        output_root / "label_mapping.json",
    )

    save_json(
        {
            "n_selected_total": len(selected_rows),
            "feature_names": feature_names,
            "n_features": len(feature_names),
            "n_classes": len(label_mapping),
            "encoding": encoding_metadata,
            "pain_feature": PAIN_FEATURE,
            "excluded_raw_fields": sorted(EXCLUDED_FIELDS),
        },
        output_root / "encoding_metadata.json",
    )

    print("Done")
    print("Patients:", len(df))
    print("Features:", len(feature_names))
    print("Classes:", len(label_mapping))
    print("Shape:", df.shape)
    print("Ordered evidence variables detected:")
    for evidence_id, info in encoding_metadata["ordered_evidence"].items():
        print(f"  {evidence_id}: {info['levels']}")


main()


Done
Patients: 20000
Features: 809
Classes: 49
Shape: (20000, 810)
Ordered evidence variables detected:
  E_59: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  E_56: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  E_58: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  E_134: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  E_136: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


### final dataset visualization

In [115]:
import json
import pandas as pd

# Load the already-cleaned CSV so this visualization cell can run independently.
clean_csv = output_root / "ddxplus.csv"
encoding_metadata_path = output_root / "encoding_metadata.json"

if not clean_csv.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found at {clean_csv}. Run the preprocessing cell first."
    )

clean_df = pd.read_csv(clean_csv)

# Load metadata when available so severity variables can be identified explicitly.
encoding_metadata = {}
if encoding_metadata_path.exists():
    with encoding_metadata_path.open("r", encoding="utf-8") as f:
        encoding_metadata = json.load(f)

feature_cols = [c for c in clean_df.columns if c != "pathology"]

print(f"Number of patients: {len(clean_df):,}")
print(f"Number of model features: {len(feature_cols)}")
print(f"Number of diagnosis classes: {clean_df['pathology'].nunique()}")
print(f"Cleaned dataset shape: {clean_df.shape}")

print("\nFirst 5 cleaned patient rows:")
display(clean_df.head())

# ---------------------------------------------------------
# Feature representation summary
# ---------------------------------------------------------
ordered_severity = encoding_metadata.get("encoding", {}).get(
    "ordered_evidence", {}
)

ordered_feature_prefixes = []
for evidence_id in ordered_severity:
    ordered_feature_prefixes.append(f"{evidence_id}_level_")

def classify_feature(name: str) -> str:
    if name == "age":
        return "Numerical"
    if name == "sex":
        return "Binary"
    if any(name.startswith(prefix) for prefix in ordered_feature_prefixes):
        return "One-hot ordinal"
    if name == "pain":
        return "Binary"
    return "One-hot categorical"

feature_info = pd.DataFrame({
    "Feature": feature_cols,
    "Type": [classify_feature(c) for c in feature_cols],
})

print("\nVerified ordered evidence variables:")
if ordered_severity:
    ordered_summary = pd.DataFrame([
        {
            "Evidence": eid,
            "Question": info["question_en"],
            "Levels": "0-10",
            "Representation": info["encoding"],
        }
        for eid, info in ordered_severity.items()
    ])
    display(ordered_summary)
else:
    print("None detected.")

print("Feature representation:")
display(feature_info)

# Compact counts by representation type.
print("Feature-type counts:")
display(feature_info["Type"].value_counts().rename_axis("Type").reset_index(name="Count"))

# ---------------------------------------------------------
# One complete patient
# ---------------------------------------------------------
patient = clean_df.iloc[0]
patient_view = pd.DataFrame({
    "Feature": feature_cols,
    "Value": patient[feature_cols].values,
    "Type": [classify_feature(c) for c in feature_cols],
})

print("Example cleaned patient:")
display(patient_view)

# Show only the severity columns for a compact demonstration.
ordered_cols = [
    c for c in feature_cols
    if classify_feature(c) == "One-hot ordinal"
]

if ordered_cols:
    print("Ordered-evidence columns for this patient:")
    display(patient_view[patient_view["Feature"].isin(ordered_cols)])


Number of patients: 20,000
Number of model features: 809
Number of diagnosis classes: 49
Cleaned dataset shape: (20000, 810)

First 5 cleaned patient rows:


,age,sex,pain,E_0,E_1,E_10,E_100,E_101,E_102,E_103,...,E_91,E_92,E_93,E_94,E_95,E_96,E_97,E_98,E_99,pathology
0,18.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9
1,4.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20
2,73.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39
3,72.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,45
4,48.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,46



Verified ordered evidence variables:


,Evidence,Question,Levels,Representation
0,E_59,How fast did the pain appear?,0-10,one-hot binary level features
1,E_56,How intense is the pain?,0-10,one-hot binary level features
2,E_58,How precisely is the pain located?,0-10,one-hot binary level features
3,E_134,How intense is the pain caused by the rash?,0-10,one-hot binary level features
4,E_136,How severe is the itching?,0-10,one-hot binary level features


Feature representation:


...

,Type,Count
0,One-hot categorical,751
1,One-hot ordinal,55
2,Binary,2
3,Numerical,1


Example cleaned patient:


,Feature,Value,Type
0,age,18.0,Numerical
1,sex,0.0,Binary
2,pain,0.0,Binary
3,E_0,0.0,One-hot categorical
4,E_1,0.0,One-hot categorical
...,...,...,...
804,E_95,0.0,One-hot categorical
805,E_96,0.0,One-hot categorical
806,E_97,0.0,One-hot categorical
807,E_98,0.0,One-hot categorical


Ordered-evidence columns for this patient:


,Feature,Value,Type
221,E_134_level_0,0.0,One-hot ordinal
222,E_134_level_1,0.0,One-hot ordinal
223,E_134_level_10,0.0,One-hot ordinal
224,E_134_level_2,0.0,One-hot ordinal
225,E_134_level_3,0.0,One-hot ordinal
226,E_134_level_4,0.0,One-hot ordinal
227,E_134_level_5,0.0,One-hot ordinal
228,E_134_level_6,0.0,One-hot ordinal
229,E_134_level_7,0.0,One-hot ordinal
230,E_134_level_8,0.0,One-hot ordinal


In [116]:
# Show patient 1's ordered/severity features
ordered_cols = [
    c for c in clean_df.columns
    if any(c.startswith(eid + "_level_") for eid in ["E_59", "E_56", "E_58", "E_134", "E_136"])
]

patient_1_severity = clean_df.loc[0, ordered_cols]

display(
    patient_1_severity.to_frame(name="Patient 1 value")
)

,Patient 1 value
E_134_level_0,0.0
E_134_level_1,0.0
E_134_level_10,0.0
E_134_level_2,0.0
E_134_level_3,0.0
E_134_level_4,0.0
E_134_level_5,0.0
E_134_level_6,0.0
E_134_level_7,0.0
E_134_level_8,0.0
